In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# --------------------------------------------------
# 1. Parameters
# --------------------------------------------------

input_size = 3
hidden_size = 4
seq_len = 5
batch_size = 2

# Fixed input sequence
x = torch.randn(batch_size, seq_len, input_size)

print("Input shape:", x.shape)


# --------------------------------------------------
# 2. Manual LSTM Cell
# --------------------------------------------------

class ManualLSTM:

    def __init__(self, input_size, hidden_size):

        self.input_size = input_size
        self.hidden_size = hidden_size

        # Input gate
        self.W_ii = torch.randn(hidden_size, input_size)
        self.W_hi = torch.randn(hidden_size, hidden_size)
        self.b_i = torch.randn(hidden_size)

        # Forget gate
        self.W_if = torch.randn(hidden_size, input_size)
        self.W_hf = torch.randn(hidden_size, hidden_size)
        self.b_f = torch.randn(hidden_size)

        # Candidate cell
        self.W_ig = torch.randn(hidden_size, input_size)
        self.W_hg = torch.randn(hidden_size, hidden_size)
        self.b_g = torch.randn(hidden_size)

        # Output gate
        self.W_io = torch.randn(hidden_size, input_size)
        self.W_ho = torch.randn(hidden_size, hidden_size)
        self.b_o = torch.randn(hidden_size)


    def step(self, x_t, h_prev, c_prev):

        # Input gate
        i_t = torch.sigmoid(
            x_t @ self.W_ii.T +
            h_prev @ self.W_hi.T +
            self.b_i
        )

        # Forget gate
        f_t = torch.sigmoid(
            x_t @ self.W_if.T +
            h_prev @ self.W_hf.T +
            self.b_f
        )

        # Candidate cell state
        g_t = torch.tanh(
            x_t @ self.W_ig.T +
            h_prev @ self.W_hg.T +
            self.b_g
        )

        # Output gate
        o_t = torch.sigmoid(
            x_t @ self.W_io.T +
            h_prev @ self.W_ho.T +
            self.b_o
        )

        # Cell state
        c_t = f_t * c_prev + i_t * g_t

        # Hidden state
        h_t = o_t * torch.tanh(c_t)

        return h_t, c_t


    def forward(self, x):

        batch_size, seq_len, _ = x.shape

        # Initial hidden and cell states
        h = torch.zeros(batch_size, self.hidden_size)
        c = torch.zeros(batch_size, self.hidden_size)

        outputs = []

        for t in range(seq_len):

            x_t = x[:, t, :]

            h, c = self.step(x_t, h, c)

            outputs.append(h)

        return torch.stack(outputs, dim=1)


# --------------------------------------------------
# 3. Create Forward and Backward LSTMs
# --------------------------------------------------

forward_lstm = ManualLSTM(input_size, hidden_size)
backward_lstm = ManualLSTM(input_size, hidden_size)


# --------------------------------------------------
# 4. Forward direction
# --------------------------------------------------

forward_output = forward_lstm.forward(x)


# --------------------------------------------------
# 5. Backward direction
# --------------------------------------------------

# Reverse sequence
x_reverse = torch.flip(x, dims=[1])

backward_reverse = backward_lstm.forward(x_reverse)

# Reverse outputs back to original order
backward_output = torch.flip(backward_reverse, dims=[1])


# --------------------------------------------------
# 6. Concatenate Forward + Backward
# --------------------------------------------------

bidirectional_output = torch.cat(
    [forward_output, backward_output],
    dim=2
)

print("\nForward output shape:")
print(forward_output.shape)

print("\nBackward output shape:")
print(backward_output.shape)

print("\nBidirectional output shape:")
print(bidirectional_output.shape)

Input shape: torch.Size([2, 5, 3])

Forward output shape:
torch.Size([2, 5, 4])

Backward output shape:
torch.Size([2, 5, 4])

Bidirectional output shape:
torch.Size([2, 5, 8])
